<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [10]</a>'.</span>

# 04 — Online PA-RL fine-tuning (Algorithm 2)

Расширение offline PA-RL на **online** режим (paper Algorithm 2):
- Загружается pre-trained critic (из 02) + base SmolVLA
- **Online buffer** накапливает rollouts текущей policy (real LIBERO sim)
- Critic **продолжает обучаться** на online transitions (IQL/Cal-QL TD)
- Policy distill на **mixed** batch (online + offline image cache)

Адаптации от paper:
- Critic updates на **mixed** batches (50% online / 50% offline) — Cal-QL practice (paper § 4.3)
- Policy distillation mixing: 50% online states + 50% offline states  
- Critic LR ниже чем в pretraining (1e-4 vs 3e-4) для anti-catastrophic-forgetting
- Cal-QL reference V используется без re-calibration (paper-faithful re-cal требует return-tracking)

Ожидаемое время на L40S: 60 шагов × ~30 сек = **~30 минут training + 4 eval × 8 мин = ~1 час total**.


## 1. Setup

In [ ]:
import os
# Reduce CUDA fragmentation для OOM на 44GB GPU (offline cache + online buffer одновременно)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["MUJOCO_GL"]="egl"; os.environ["PYOPENGL_PLATFORM"]="egl"

import sys, math, time, gc, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
device = torch.device("cuda:0")
torch.backends.cudnn.benchmark = True

MODEL_ID            = "HuggingFaceVLA/smolvla_libero"
CRITIC_PATH         = "/workspace/out/critic_resnet_012.pt"
SAVE_VLA_PATH       = "/workspace/out/smolvla_parl_online"

TASK_DESCRIPTION = "pick up the black bowl between the plate and the ramekin and place it on the plate"

TASK_SUITE_NAME       = "libero_spatial"
TASK_ID               = 0
ENV_IMAGE_SIZE        = 256
SEED                  = 43
MAX_STEPS             = 90
NUM_STABILIZATION_STEPS = 10
INIT_STATES_IDS       = [1, 2, 6, 7, 13, 22, 23, 27, 32, 35, 38, 47, 46]  # canonical eval

# === PA-RL hyperparameters (paper App B.1 + наши adjustments) ===
EVAL_AT_STEPS      = [20, 40, 60, 80, 100, 120, 140, 160]
PARL_STEPS         = max(EVAL_AT_STEPS)
BATCH_SIZE         = 32
N_CANDIDATES       = 8
M_GLOBAL           = 4
N_REFINEMENT_STEPS = 5
ETA_LOCAL          = 3e-4
NUM_DENOISE_STEPS  = 4
POLICY_LR          = 1e-5
LOG_EVERY          = 5
LORA_L2_COEF       = 0.001

# === ONLINE-specific hyperparameters ===
ROLLOUT_EVERY                = 10       # каждые 5 distill steps (было 10) — fresher policy data
N_ROLLOUTS_PER_COLLECTION    = 5       # 5 rollout-ов за раз (~2.5 мин на сбор)
CRITIC_UPDATES_PER_DISTILL   = 1       # 1 critic update per distill step (было 3 — agressive drift)
MIXING_RATIO_DISTILL         = 0.5     # 50% online / 50% offline для policy distillation
ONLINE_BUFFER_CAPACITY       = 5000    # FIFO capacity (transitions)
# ~3.1 GB на GPU; за 60 шагов online run собирает ~2700 frames, capacity = вмещает всё + запас.
# Раньше пробовали 5000 → OOM, 1500 → буфер переполнялся выкидывал свои же недавние rollouts.

# Critic / IQL hyperparameters (same как в 02_critic_train но LR ниже)
CRITIC_LR_ONLINE   = 1e-4    # понижено с 1e-4 — был critic drift, td_loss spike до 9.07, SR degraded
V_LR_ONLINE        = 1e-5
GAMMA              = 0.99
EXPECTILE          = 0.7
CAL_ALPHA          = 0.005
TARGET_TAU         = 0.001  # медленнее target sync (было 0.005) — стабильность online

torch.manual_seed(SEED); np.random.seed(SEED)

from huggingface_hub import login
login(token="", add_to_git_credential=False)
print("[hf] logged in")

print(f"\nONLINE settings:")
print(f"  ROLLOUT_EVERY              = {ROLLOUT_EVERY}")
print(f"  N_ROLLOUTS_PER_COLLECTION  = {N_ROLLOUTS_PER_COLLECTION}")
print(f"  CRITIC_UPDATES_PER_DISTILL = {CRITIC_UPDATES_PER_DISTILL}")
print(f"  MIXING_RATIO_DISTILL       = {MIXING_RATIO_DISTILL}")
print(f"  ONLINE_BUFFER_CAPACITY     = {ONLINE_BUFFER_CAPACITY}")
print(f"  CRITIC_LR_ONLINE           = {CRITIC_LR_ONLINE}")


PyTorch: 2.11.0+cu130, CUDA: True


[hf] logged in

ONLINE settings:
  ROLLOUT_EVERY              = 10
  N_ROLLOUTS_PER_COLLECTION  = 5
  CRITIC_UPDATES_PER_DISTILL = 1
  MIXING_RATIO_DISTILL       = 0.5
  ONLINE_BUFFER_CAPACITY     = 5000
  CRITIC_LR_ONLINE           = 0.0001


## 2. Загрузка SmolVLA + LoRA

In [2]:
from lerobot.policies.smolvla.modeling_smolvla import (
    SmolVLAPolicy, resize_with_pad, pad_vector, make_att_2d_masks
)
from lerobot.policies.factory import make_pre_post_processors

policy = SmolVLAPolicy.from_pretrained(MODEL_ID).to(device)
preprocessor, postprocessor = make_pre_post_processors(
    policy_cfg=policy.config, pretrained_path=MODEL_ID,
    preprocessor_overrides={"device_processor": {"device": str(device)}},
)
print(f"Policy loaded, chunk_size={policy.config.chunk_size}, num_steps={policy.config.num_steps}")

# LoRA wrapping
n_total = sum(p.numel() for p in policy.parameters())
for p in policy.parameters():
    p.requires_grad_(False)


class LoRALinear(nn.Module):
    def __init__(self, base_layer: nn.Linear, rank: int = 32, alpha: float = 8.0):
        super().__init__()
        self.base = base_layer
        for p in self.base.parameters():
            p.requires_grad_(False)
        in_f = base_layer.in_features
        out_f = base_layer.out_features
        self.in_features = in_f
        self.out_features = out_f
        self.lora_A = nn.Linear(in_f, rank, bias=False)
        self.lora_B = nn.Linear(rank, out_f, bias=False)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)
        self.scaling = alpha / rank
    @property
    def weight(self): return self.base.weight
    @property
    def bias(self):   return self.base.bias
    def forward(self, x):
        base_out = self.base(x)
        x_lora = x.to(self.lora_A.weight.dtype)
        lora_out = self.scaling * self.lora_B(self.lora_A(x_lora))
        return base_out + lora_out.to(base_out.dtype)


def apply_lora(module, target_names, rank=32, alpha=8):
    n_replaced = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and any(t in name for t in target_names):
            setattr(module, name, LoRALinear(child, rank=rank, alpha=alpha))
            n_replaced += 1
        else:
            n_replaced += apply_lora(child, target_names, rank, alpha)
    return n_replaced


LORA_RANK = 32
LORA_ALPHA = 8
target_names = ["q_proj", "k_proj", "v_proj", "o_proj"]

if hasattr(policy.model.vlm_with_expert, "lm_expert"):
    n_lora = apply_lora(policy.model.vlm_with_expert.lm_expert, target_names,
                        rank=LORA_RANK, alpha=LORA_ALPHA)
    print(f"Applied LoRA to {n_lora} attention projections in lm_expert")

policy.to(device)
n_trainable = sum(p.numel() for p in policy.parameters() if p.requires_grad)
print(f"Trainable (LoRA only): {n_trainable/1e6:.2f}M ({100*n_trainable/n_total:.2f}%)")

NUM_DENOISE_STEPS = policy.config.num_steps
print(f"NUM_DENOISE_STEPS = {NUM_DENOISE_STEPS}")


`torch_dtype` is deprecated! Use `dtype` instead!


Loading  HuggingFaceTB/SmolVLM2-500M-Instruct weights ...


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

Policy loaded, chunk_size=50, num_steps=10
Applied LoRA to 128 attention projections in lm_expert
Trainable (LoRA only): 4.42M (0.73%)
NUM_DENOISE_STEPS = 10


## 3. Загрузка critic — НЕ замораживаем

In [3]:
from torchvision.models import resnet18

class ImageEncoder(nn.Module):
    def __init__(self, out_dim=512):
        super().__init__()
        self.backbone = resnet18(weights=None)
        self.backbone.fc = nn.Identity()
        self.out_dim = 512
    def forward(self, img):
        return self.backbone(img)


class CriticHead(nn.Module):
    def __init__(self, obs_dim, action_dim=7, hidden=512, action_emb_dim=128):
        super().__init__()
        self.action_emb = nn.Sequential(
            nn.Linear(action_dim, action_emb_dim), nn.LayerNorm(action_emb_dim), nn.ReLU(),
            nn.Linear(action_emb_dim, action_emb_dim), nn.LayerNorm(action_emb_dim), nn.ReLU(),
        )
        self.l1 = nn.Sequential(nn.Linear(obs_dim+action_emb_dim, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.l2 = nn.Sequential(nn.Linear(hidden+action_emb_dim, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.l3 = nn.Sequential(nn.Linear(hidden+action_emb_dim, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.l4 = nn.Linear(hidden+action_emb_dim, 1)
    def forward(self, obs, a):
        a_emb = self.action_emb(a)
        x = self.l1(torch.cat([obs, a_emb], -1))
        x = self.l2(torch.cat([x, a_emb], -1))
        x = self.l3(torch.cat([x, a_emb], -1))
        return self.l4(torch.cat([x, a_emb], -1)).squeeze(-1)


class CriticEnsemble(nn.Module):
    def __init__(self, state_dim=8, action_dim=7, hidden=512, state_hidden=64):
        super().__init__()
        self.enc1 = ImageEncoder()
        self.enc2 = ImageEncoder()
        self.state_mlp = nn.Sequential(
            nn.Linear(state_dim, state_hidden), nn.LayerNorm(state_hidden), nn.ReLU(),
            nn.Linear(state_hidden, state_hidden),
        )
        self.obs_dim = 512 + 512 + state_hidden
        self.obs_norm = nn.LayerNorm(self.obs_dim)
        self.q1 = CriticHead(self.obs_dim, action_dim, hidden)
        self.q2 = CriticHead(self.obs_dim, action_dim, hidden)
    def encode(self, img1, img2, state):
        e1 = self.enc1(img1)
        e2 = self.enc2(img2)
        s  = self.state_mlp(state)
        obs = torch.cat([e1, e2, s], dim=-1)
        return self.obs_norm(obs)


class VNetwork(nn.Module):
    def __init__(self, obs_dim, hidden=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, obs):
        return self.net(obs).squeeze(-1)


critic = CriticEnsemble().to(device)
v_net  = VNetwork(critic.obs_dim).to(device)

state = torch.load(CRITIC_PATH, map_location=device, weights_only=False)
critic.load_state_dict(state["critic"])
v_net.load_state_dict(state["v_net"])
state_mean = state["state_mean"]
state_std  = state["state_std"]

# UNFREEZE — критик будет обновляться online
for p in critic.parameters(): p.requires_grad_(True)
for p in v_net.parameters():  p.requires_grad_(True)
critic.train(); v_net.train()

# Target critic для Polyak update
critic_target = copy.deepcopy(critic)
for p in critic_target.parameters(): p.requires_grad_(False)
critic_target.eval()

# Optimizers для online critic
critic_optim_online = torch.optim.Adam(critic.parameters(), lr=CRITIC_LR_ONLINE)
v_optim_online      = torch.optim.Adam(v_net.parameters(),  lr=V_LR_ONLINE)

print(f"Critic + V loaded из {CRITIC_PATH}")
print(f"  Critic params: {sum(p.numel() for p in critic.parameters())/1e6:.1f}M (TRAINABLE)")
print(f"  V params:      {sum(p.numel() for p in v_net.parameters())/1e6:.1f}M (TRAINABLE)")
print(f"  target_critic created (frozen, Polyak τ={TARGET_TAU})")


# Освободить fragmentation после загрузки критика
import gc
gc.collect(); torch.cuda.empty_cache()
print(f"After critic load — GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated, "
      f"{torch.cuda.memory_reserved()/1e9:.1f} GB reserved")


Critic + V loaded из /workspace/out/critic_resnet_012.pt
  Critic params: 25.0M (TRAINABLE)
  V params:      1.1M (TRAINABLE)
  target_critic created (frozen, Polyak τ=0.001)
After critic load — GPU: 1.6 GB allocated, 1.7 GB reserved


## 4. Offline image cache (для PA-RL training mixing)

In [4]:
CRITIC_IMG_SIZE = 128
RAW_IMG_SIZE    = 256
IMG_CACHE_PATH  = "/workspace/data/img_cache_critic_256.npz"

print(f"Загружаем image cache {IMG_CACHE_PATH}…")
data = np.load(IMG_CACHE_PATH)
img1_cache = data["img1"]
img2_cache = data["img2"]
state_cache = data["state"].astype(np.float32)
action_cache = data["action"].astype(np.float32)
episode_cache = data["episode"].astype(np.int64)
frame_cache = data["frame"].astype(np.int64)
print(f"  loaded: {img1_cache.shape} ({img1_cache.nbytes/1e9:.1f} GB)")

state_mean_np = state_mean if isinstance(state_mean, np.ndarray) else state_mean.cpu().numpy()
state_std_np  = state_std  if isinstance(state_std,  np.ndarray) else state_std.cpu().numpy()
state_norm_cache = (state_cache - state_mean_np) / state_std_np

# Sort by (episode, frame) — критично для построения s' через i+1
sort_order = np.lexsort((frame_cache, episode_cache))
img1_sorted        = img1_cache[sort_order]
img2_sorted        = img2_cache[sort_order]
state_norm_sorted  = state_norm_cache[sort_order]
state_raw_sorted   = state_cache[sort_order]
action_sorted      = action_cache[sort_order]
episode_sorted     = episode_cache[sort_order]
frame_sorted       = frame_cache[sort_order]
del img1_cache, img2_cache
gc.collect()

print("Перенос на GPU…")
img1_gpu      = torch.from_numpy(img1_sorted).to(device)
img2_gpu      = torch.from_numpy(img2_sorted).to(device)
state_gpu     = torch.from_numpy(state_norm_sorted).to(device)
state_raw_gpu = torch.from_numpy(state_raw_sorted).to(device)
action_gpu    = torch.from_numpy(action_sorted).to(device)
print(f"  GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")
N_OFFLINE = img1_gpu.shape[0]
print(f"  N offline frames: {N_OFFLINE}")


def downscale_only(img, target_size=CRITIC_IMG_SIZE):
    if target_size != img.shape[-1]:
        return F.interpolate(img, size=target_size, mode='bilinear', align_corners=False)
    return img


chunk_size = policy.config.chunk_size
state_mean_t = torch.tensor(state_mean_np, device=device, dtype=torch.float32)
state_std_t  = torch.tensor(state_std_np, device=device, dtype=torch.float32)


# ─── Offline transitions (r, s', done) для critic training ───
# Cal-QL practice (paper § 4.3): critic train на mixture of offline + online
# Reward shaping (same as 02_critic_train):
#   r = -1 каждый шаг, кроме success terminal (r=0)
#   done=1 на последнем кадре эпизода
# img_cache_critic_256.npz содержит ТОЛЬКО demos (все success) — все last frames = success

is_last_in_ep_np = np.zeros(N_OFFLINE, dtype=bool)
for i in range(N_OFFLINE - 1):
    if episode_sorted[i] != episode_sorted[i+1]:
        is_last_in_ep_np[i] = True
is_last_in_ep_np[-1] = True

rewards_np = np.full(N_OFFLINE, -1.0, dtype=np.float32)
rewards_np[is_last_in_ep_np] = 0.0  # success terminal
dones_np = is_last_in_ep_np.astype(np.float32)

# Next-state index: для не-last это i+1, для last это сам i (done=1 → s' не используется в bootstrap)
next_idx_np = np.arange(N_OFFLINE) + 1
next_idx_np[is_last_in_ep_np] = np.arange(N_OFFLINE)[is_last_in_ep_np]
next_idx_np = np.clip(next_idx_np, 0, N_OFFLINE - 1)

reward_gpu   = torch.from_numpy(rewards_np).to(device)
done_gpu     = torch.from_numpy(dones_np).to(device)
next_idx_gpu = torch.from_numpy(next_idx_np).to(device)

n_terminals = int(is_last_in_ep_np.sum())
print(f"  Offline transitions built: {n_terminals} terminal frames (success), "
      f"{N_OFFLINE - n_terminals} non-terminal")


def sample_offline_batch_for_distill(batch_size):
    """Семплирует batch из offline cache для policy distillation."""
    idx = torch.randint(0, N_OFFLINE - chunk_size, (batch_size,), device=device)
    img1_critic = downscale_only(img1_gpu[idx].permute(0, 3, 1, 2).float() / 255.0)
    img2_critic = downscale_only(img2_gpu[idx].permute(0, 3, 1, 2).float() / 255.0)
    state_critic = state_gpu[idx]
    img1_policy = img1_gpu[idx].permute(0, 3, 1, 2).float() / 255.0
    img2_policy = img2_gpu[idx].permute(0, 3, 1, 2).float() / 255.0
    state_raw = state_gpu[idx] * state_std_t + state_mean_t
    action_demo = action_gpu[idx]
    return {
        "img1_critic": img1_critic, "img2_critic": img2_critic, "state_critic": state_critic,
        "img1_policy": img1_policy, "img2_policy": img2_policy, "state_raw": state_raw,
        "action_demo": action_demo,
    }


def sample_offline_batch_for_critic(batch_size):
    """Семплирует batch (s, a, r, s', done) из offline cache для IQL/Cal-QL critic update."""
    idx = torch.randint(0, N_OFFLINE, (batch_size,), device=device)
    nidx = next_idx_gpu[idx]
    return {
        "img1_curr":  downscale_only(img1_gpu[idx].permute(0, 3, 1, 2).float() / 255.0),
        "img2_curr":  downscale_only(img2_gpu[idx].permute(0, 3, 1, 2).float() / 255.0),
        "state_curr": state_gpu[idx],
        "action":     action_gpu[idx],
        "reward":     reward_gpu[idx],
        "done":       done_gpu[idx],
        "img1_next":  downscale_only(img1_gpu[nidx].permute(0, 3, 1, 2).float() / 255.0),
        "img2_next":  downscale_only(img2_gpu[nidx].permute(0, 3, 1, 2).float() / 255.0),
        "state_next": state_gpu[nidx],
    }


Загружаем image cache /workspace/data/img_cache_critic_256.npz…


  loaded: (52970, 256, 256, 3) (10.4 GB)


Перенос на GPU…


  GPU memory: 22.5 GB
  N offline frames: 52970
  Offline transitions built: 432 terminal frames (success), 52538 non-terminal


## 5. OnlineBuffer — GPU FIFO для online transitions

In [5]:
class OnlineBuffer:
    """GPU-resident FIFO buffer для online IQL transitions.
    
    Каждый transition: (s, a, r, s', done) где s = (img1, img2, state).
    Память: capacity × 2 cameras × 2 (curr+next) × 256² × 3 uint8 ≈ 4 GB для capacity=5000.
    """
    def __init__(self, capacity=5000, image_size=RAW_IMG_SIZE, state_dim=8, action_dim=7,
                 device='cuda'):
        self.capacity = capacity
        self.device   = device
        self.size     = 0
        self.ptr      = 0
        
        self.img1_curr  = torch.zeros(capacity, image_size, image_size, 3, dtype=torch.uint8, device=device)
        self.img2_curr  = torch.zeros(capacity, image_size, image_size, 3, dtype=torch.uint8, device=device)
        self.state_curr = torch.zeros(capacity, state_dim, dtype=torch.float32, device=device)  # RAW (unnorm)
        self.action     = torch.zeros(capacity, action_dim, dtype=torch.float32, device=device)
        self.reward     = torch.zeros(capacity, dtype=torch.float32, device=device)
        self.done       = torch.zeros(capacity, dtype=torch.float32, device=device)
        self.img1_next  = torch.zeros(capacity, image_size, image_size, 3, dtype=torch.uint8, device=device)
        self.img2_next  = torch.zeros(capacity, image_size, image_size, 3, dtype=torch.uint8, device=device)
        self.state_next = torch.zeros(capacity, state_dim, dtype=torch.float32, device=device)
        
        mem_gb = (image_size * image_size * 3 * 4 * capacity) / 1e9
        print(f"OnlineBuffer: capacity={capacity}, image memory ~{mem_gb:.2f} GB")
    
    def add_transition(self, img1, img2, state, action, reward, done,
                       img1_next, img2_next, state_next):
        """Все аргументы — torch tensors на device."""
        i = self.ptr
        self.img1_curr[i]  = img1
        self.img2_curr[i]  = img2
        self.state_curr[i] = state
        self.action[i]     = action
        self.reward[i]     = reward
        self.done[i]       = done
        self.img1_next[i]  = img1_next
        self.img2_next[i]  = img2_next
        self.state_next[i] = state_next
        self.ptr  = (self.ptr + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)
    
    def sample(self, batch_size):
        """Возвращает batch уже-normalized states + downscaled images, готовые для IQL."""
        idx = torch.randint(0, self.size, (batch_size,), device=self.device)
        img1_curr = self.img1_curr[idx].permute(0, 3, 1, 2).float() / 255.0  # [B, 3, 256, 256]
        img2_curr = self.img2_curr[idx].permute(0, 3, 1, 2).float() / 255.0
        img1_next = self.img1_next[idx].permute(0, 3, 1, 2).float() / 255.0
        img2_next = self.img2_next[idx].permute(0, 3, 1, 2).float() / 255.0
        state_curr_norm = (self.state_curr[idx] - state_mean_t) / state_std_t
        state_next_norm = (self.state_next[idx] - state_mean_t) / state_std_t
        return {
            "img1_curr":   downscale_only(img1_curr),
            "img2_curr":   downscale_only(img2_curr),
            "state_curr":  state_curr_norm,
            "action":      self.action[idx],
            "reward":      self.reward[idx],
            "done":        self.done[idx],
            "img1_next":   downscale_only(img1_next),
            "img2_next":   downscale_only(img2_next),
            "state_next":  state_next_norm,
            # Для distill — full-res images + raw state
            "img1_policy": img1_curr,
            "img2_policy": img2_curr,
            "state_raw":   self.state_curr[idx],
            "action_demo": self.action[idx],
            "img1_critic": downscale_only(img1_curr),
            "img2_critic": downscale_only(img2_curr),
            "state_critic": state_curr_norm,
        }


online_buffer = OnlineBuffer(capacity=ONLINE_BUFFER_CAPACITY)


OnlineBuffer: capacity=5000, image memory ~3.93 GB


## 6. Сбор online rollouts текущей policy

In [6]:
import builtins
builtins.input = lambda _: "n"
os.environ["LIBERO_DATA_PATH"] = "/workspace/libero_data"
os.makedirs("/workspace/libero_data", exist_ok=True)

from libero.libero import get_libero_path, benchmark
from libero.libero.envs import OffScreenRenderEnv

benchmark_dict = benchmark.get_benchmark_dict()
task_suite = benchmark_dict[TASK_SUITE_NAME]()
task = task_suite.get_task(TASK_ID)
task_description_libero = task.language
task_bddl_file = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)

env = OffScreenRenderEnv(bddl_file_name=task_bddl_file,
                         camera_heights=ENV_IMAGE_SIZE, camera_widths=ENV_IMAGE_SIZE)
env.seed(SEED)
init_states = task_suite.get_task_init_states(TASK_ID)
print(f"Task: {task_description_libero}")

# Доступные init_states для rollout collection (исключая eval set)
available_state_ids = [i for i in range(len(init_states)) if i not in INIT_STATES_IDS]
print(f"Available init_states для online rollouts: {len(available_state_ids)}")


def quat2axisangle(quat):
    quat = quat.astype(np.float32).copy()
    quat[3] = np.clip(quat[3], -1.0, 1.0)
    den = np.sqrt(max(1e-12, 1.0 - quat[3]**2))
    if np.isclose(den, 0.0): return np.zeros(3, dtype=np.float32)
    angle = 2.0 * math.acos(float(quat[3]))
    return ((quat[:3] * angle) / den).astype(np.float32)


def rotate_180(im):
    return np.ascontiguousarray(im[::-1, ::-1])


def libero_obs_to_critic_raw(raw_obs):
    """Извлекает (img1, img2, state) как torch tensors на device. Images uint8, state float32 RAW."""
    a_img = rotate_180(raw_obs["agentview_image"]).astype(np.uint8)
    w_img = rotate_180(raw_obs["robot0_eye_in_hand_image"]).astype(np.uint8)
    eef_pos = raw_obs["robot0_eef_pos"].astype(np.float32)
    eef_quat = raw_obs["robot0_eef_quat"].astype(np.float32)
    grip = raw_obs["robot0_gripper_qpos"].astype(np.float32)
    state = np.concatenate([eef_pos, quat2axisangle(eef_quat), grip], 0).astype(np.float32)
    return (
        torch.from_numpy(a_img).to(device),       # [256, 256, 3] uint8
        torch.from_numpy(w_img).to(device),       # [256, 256, 3] uint8
        torch.from_numpy(state).to(device),       # [8] float32 RAW
    )


def libero_obs_to_lerobot(raw_obs, task_text):
    a_img = rotate_180(raw_obs["agentview_image"])
    w_img = rotate_180(raw_obs["robot0_eye_in_hand_image"])
    eef_pos = raw_obs["robot0_eef_pos"].astype(np.float32)
    eef_quat = raw_obs["robot0_eef_quat"].astype(np.float32)
    grip = raw_obs["robot0_gripper_qpos"].astype(np.float32)
    state = np.concatenate([eef_pos, quat2axisangle(eef_quat), grip], 0).astype(np.float32)
    return {
        "observation.images.image":  torch.from_numpy(a_img).permute(2,0,1).unsqueeze(0).float()/255.0,
        "observation.images.image2": torch.from_numpy(w_img).permute(2,0,1).unsqueeze(0).float()/255.0,
        "observation.state":         torch.from_numpy(state).unsqueeze(0).float(),
        "task":                      [task_text],
    }


@torch.no_grad()
def predict_action_for_rollout(raw_obs, task_text):
    obs = libero_obs_to_lerobot(raw_obs, task_text)
    obs = preprocessor(obs)
    a = policy.select_action(obs)
    a = postprocessor(a)
    return a.squeeze(0).detach().cpu().numpy().astype(np.float32)


def collect_online_rollouts(n_rollouts, buffer):
    """Запускает policy в env, собирает (s, a, r, s', d) и кладёт в buffer.
    
    Reward shaping (как в 02_critic_train):
    - r = -1 на каждом шаге, кроме терминала
    - Success terminal (env.done=True): r = 0, done = 1
    - Failure terminal (timeout): r = -1, done = 1
    """
    policy.eval()
    n_succ = 0
    n_total_frames = 0
    
    for r_idx in range(n_rollouts):
        state_id = np.random.choice(available_state_ids)
        policy.reset()
        env.reset()
        raw_obs = env.set_init_state(init_states[state_id])
        
        # Накопление transitions перед добавлением в buffer
        transitions = []  # list of dicts
        prev_obs = None
        prev_action = None
        success = False
        
        for t in range(MAX_STEPS + NUM_STABILIZATION_STEPS):
            if t < NUM_STABILIZATION_STEPS:
                action_list = [0.0]*6 + [-1.0]
            else:
                a_np = predict_action_for_rollout(raw_obs, task_description_libero)
                action_list = a_np.tolist()
                # Сохраняем CURRENT obs + action (для transition)
                img1_t, img2_t, state_t = libero_obs_to_critic_raw(raw_obs)
                prev_obs = (img1_t, img2_t, state_t)
                prev_action = torch.from_numpy(a_np[:7].astype(np.float32)).to(device)
            
            raw_obs, _, done, _ = env.step(action_list)
            
            # После step — есть NEXT obs. Записать transition если есть prev
            if t >= NUM_STABILIZATION_STEPS and prev_obs is not None:
                img1_next, img2_next, state_next = libero_obs_to_critic_raw(raw_obs)
                # Reward, done
                if done:
                    r_val = 0.0  # success
                    d_val = 1.0
                    success = True
                else:
                    r_val = -1.0
                    d_val = 0.0
                
                transitions.append({
                    "img1":       prev_obs[0],
                    "img2":       prev_obs[1],
                    "state":      prev_obs[2],
                    "action":     prev_action,
                    "reward":     torch.tensor(r_val, device=device),
                    "done":       torch.tensor(d_val, device=device),
                    "img1_next":  img1_next,
                    "img2_next":  img2_next,
                    "state_next": state_next,
                })
                prev_obs = None
                prev_action = None
            
            if done:
                break
        
        # Если эпизод timeout-нулся (не сработал done) — последний transition должен быть failure
        if not success and transitions:
            # Mark last transition as failure terminal
            transitions[-1]["reward"] = torch.tensor(-1.0, device=device)
            transitions[-1]["done"]   = torch.tensor(1.0, device=device)
        
        # Дописываем все transitions в buffer
        for trans in transitions:
            buffer.add_transition(
                trans["img1"], trans["img2"], trans["state"], trans["action"],
                trans["reward"], trans["done"],
                trans["img1_next"], trans["img2_next"], trans["state_next"],
            )
        
        if success: n_succ += 1
        n_total_frames += len(transitions)
    
    policy.train()
    return n_succ, n_total_frames


[robosuite WARNING] No private macro file found! (__init__.py:7)


[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)


[robosuite WARNING] To setup, run: python /venv/main/lib/python3.12/site-packages/robosuite/scripts/setup_macros.py (__init__.py:9)


[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Local assets not found. Downloading from HuggingFace Hub...
Assets already downloaded at /root/.cache/libero/assets


Task: pick up the black bowl between the plate and the ramekin and place it on the plate
Available init_states для online rollouts: 37


## 7. Online critic update (IQL + Cal-QL TD)

In [7]:
def expectile_loss(diff, expectile):
    weight = torch.where(diff > 0, expectile, 1 - expectile)
    return weight * (diff ** 2)


def online_critic_step(batch):
    """Один шаг IQL + Cal-QL update критика на online transitions.
    
    Идентично 02_critic_train iql_step, но:
    - Использует critic_optim_online / v_optim_online (lower LR)
    - На online buffer данных
    """
    img1c   = batch["img1_curr"]
    img2c   = batch["img2_curr"]
    img1n   = batch["img1_next"]
    img2n   = batch["img2_next"]
    state_c = batch["state_curr"]
    state_n = batch["state_next"]
    action  = batch["action"]
    reward  = batch["reward"]
    done    = batch["done"]
    
    # Forward
    with torch.no_grad():
        obs_curr_target = critic_target.encode(img1c, img2c, state_c)
        obs_next_target = critic_target.encode(img1n, img2n, state_n)
    obs_curr = critic.encode(img1c, img2c, state_c)
    
    # ── V update (expectile loss) ──
    with torch.no_grad():
        q1_t = critic_target.q1(obs_curr_target, action)
        q2_t = critic_target.q2(obs_curr_target, action)
        q_target_val = torch.minimum(q1_t, q2_t)
    v_val = v_net(obs_curr_target)
    v_loss = expectile_loss(q_target_val - v_val, EXPECTILE).mean()
    v_optim_online.zero_grad(); v_loss.backward(); v_optim_online.step()
    
    # ── Q update (TD + Cal-QL) ──
    with torch.no_grad():
        next_v = v_net(obs_next_target)
        bellman_target = reward + GAMMA * (1.0 - done) * next_v
    
    q1 = critic.q1(obs_curr, action)
    q2 = critic.q2(obs_curr, action)
    td_loss = F.mse_loss(q1, bellman_target) + F.mse_loss(q2, bellman_target)
    
    # Cal-QL
    a_random = torch.rand_like(action) * 2 - 1
    q1_rand = critic.q1(obs_curr, a_random)
    q2_rand = critic.q2(obs_curr, a_random)
    q_rand = torch.minimum(q1_rand, q2_rand)
    q_demo_avg = (q1 + q2) / 2
    
    with torch.no_grad():
        v_ref = v_net(obs_curr.detach())
    cal_reg = (torch.maximum(q_rand, v_ref) - q_demo_avg.detach()).mean()
    
    q_loss = td_loss + CAL_ALPHA * cal_reg
    critic_optim_online.zero_grad(); q_loss.backward(); critic_optim_online.step()
    
    # Polyak target update
    with torch.no_grad():
        for p, pt in zip(critic.parameters(), critic_target.parameters()):
            pt.data.mul_(1 - TARGET_TAU).add_(p.data, alpha=TARGET_TAU)
    
    return {
        "td_loss": td_loss.item(),
        "v_loss":  v_loss.item(),
        "cal_reg": cal_reg.item(),
        "q_demo":  q_demo_avg.mean().item(),
        "q_rand":  q_rand.mean().item(),
    }


## 8. PA-RL step (sample → global → local → distill)

In [8]:
@torch.no_grad()
def prepare_policy_inputs(img1_policy, img2_policy, state_raw, batch_size):
    sp = preprocessor({
        "observation.images.image":       torch.zeros(batch_size, 3, 4, 4),
        "observation.images.wrist_image": torch.zeros(batch_size, 3, 4, 4),
        "observation.state":              state_raw.cpu(),
        "task":                           [TASK_DESCRIPTION] * batch_size,
    })
    state_norm  = sp["observation.state"].to(device)
    lang_tokens = sp["observation.language.tokens"].to(device)
    lang_masks  = sp["observation.language.attention_mask"].to(device)
    raw = {
        "observation.images.image":            img1_policy,
        "observation.images.image2":           img2_policy,
        "observation.images.wrist_image":      img2_policy,
        "observation.state":                   state_norm,
        "observation.language.tokens":         lang_tokens,
        "observation.language.attention_mask": lang_masks,
    }
    images, img_masks = policy.prepare_images(raw)
    state_p           = policy.prepare_state(raw)
    return raw, images, img_masks, state_p, lang_tokens, lang_masks


@torch.no_grad()
def sample_n_actions(images, img_masks, state_p, lang_tokens, lang_masks,
                     n_candidates=None):
    if n_candidates is None: n_candidates = N_CANDIDATES
    B = images[0].shape[0] if isinstance(images, list) else images.shape[0]
    
    prefix_embs, prefix_pad_masks, prefix_att_masks = policy.model.embed_prefix(
        images, img_masks, lang_tokens, lang_masks, state=state_p
    )
    prefix_att_2d  = make_att_2d_masks(prefix_pad_masks, prefix_att_masks)
    prefix_pos_ids = torch.cumsum(prefix_pad_masks, dim=1) - 1
    _, past_kv = policy.model.vlm_with_expert.forward(
        attention_mask=prefix_att_2d, position_ids=prefix_pos_ids,
        past_key_values=None, inputs_embeds=[prefix_embs, None],
        use_cache=True, fill_kv_cache=True,
    )
    
    actions_shape = (B, policy.config.chunk_size, policy.config.max_action_dim)
    dt = -1.0 / NUM_DENOISE_STEPS
    
    chunks = []
    for _ in range(n_candidates):
        x_t = policy.model.sample_noise(actions_shape, device)
        for step in range(NUM_DENOISE_STEPS):
            t_val = 1.0 + step * dt
            t_tensor = torch.tensor(t_val, device=device).expand(B)
            v_t = policy.model.denoise_step(prefix_pad_masks, past_kv, x_t, t_tensor)
            x_t = x_t + dt * v_t
        chunks.append(x_t[:, :, :7])
    return torch.stack(chunks, dim=1)


def pa_rl_step(batch):
    """Один шаг PA-RL distillation (sample N → global → local → distill BC)."""
    img1_critic  = batch["img1_critic"]
    img2_critic  = batch["img2_critic"]
    state_critic = batch["state_critic"]
    img1_policy  = batch["img1_policy"]
    img2_policy  = batch["img2_policy"]
    state_raw    = batch["state_raw"]
    action_demo  = batch["action_demo"]
    B = img1_critic.shape[0]
    
    # Prepare inputs
    raw, images, img_masks, state_p, lang_tokens, lang_masks = \
        prepare_policy_inputs(img1_policy, img2_policy, state_raw, B)
    
    # Sample N chunks in eval mode
    policy.eval()
    candidates_full = sample_n_actions(images, img_masks, state_p, lang_tokens, lang_masks,
                                       n_candidates=N_CANDIDATES)  # [B, N, chunk, 7]
    policy.train()
    
    # Critic encode (no grad)
    with torch.no_grad():
        obs_critic = critic.encode(img1_critic, img2_critic, state_critic)
    
    # Global rerank
    with torch.no_grad():
        c_first = candidates_full[:, :, 0, :].contiguous()
        obs_exp = obs_critic.unsqueeze(1).expand(-1, N_CANDIDATES, -1)
        q_all = critic.q1(
            obs_exp.reshape(B * N_CANDIDATES, -1),
            c_first.reshape(B * N_CANDIDATES, 7),
        ).reshape(B, N_CANDIDATES)
        topm_q, topm_idx = q_all.topk(M_GLOBAL, dim=1)
        b_idx = torch.arange(B, device=device).unsqueeze(1).expand(-1, M_GLOBAL)
        topm_chunks = candidates_full[b_idx, topm_idx]
        topm_first  = topm_chunks[:, :, 0, :].contiguous()
        q_best_mean = q_all.max(dim=1).values.mean().item()
    
    # Local refine
    K_init = M_GLOBAL + 1
    a_init = torch.cat([topm_first, action_demo.unsqueeze(1)], dim=1)
    a_flat = a_init.reshape(B * K_init, 7)
    obs_K  = obs_critic.unsqueeze(1).expand(-1, K_init, -1).reshape(B * K_init, -1)
    
    gripper_init = a_flat[:, 6:7].clone()
    a = a_flat.detach().clone()
    sticky = torch.ones(B * K_init, dtype=torch.bool, device=device)
    for _ in range(N_REFINEMENT_STEPS):
        a_req = a.detach().requires_grad_(True)
        q     = critic.q1(obs_K, a_req)
        g     = torch.autograd.grad(q.sum(), a_req)[0]
        g[:, 6] = 0.0
        a_new = (a_req + ETA_LOCAL * g).clamp(-1, 1).detach()
        a_new[:, 6] = gripper_init.squeeze(-1)
        with torch.no_grad():
            q_new = critic.q1(obs_K, a_new)
            improved = (q_new > q.detach())
            sticky   = sticky & improved
            a        = torch.where(sticky.unsqueeze(-1), a_new, a)
    
    refined = a.detach().reshape(B, K_init, 7)
    
    # Categorical pick (Eq 4.3)
    with torch.no_grad():
        all_q = critic.q1(obs_K, refined.reshape(B * K_init, 7)).reshape(B, K_init)
        q_top_mean = all_q.max(dim=1).values.mean().item()
        q_std = all_q.std(dim=1).mean().item()
        if q_std < 0.1:
            best_idx = all_q.argmax(dim=1)
            a_first  = refined[torch.arange(B), best_idx]
        else:
            weights = torch.softmax(all_q, dim=1)
            sampled = torch.multinomial(weights, num_samples=1).squeeze(-1)
            a_first = refined[torch.arange(B), sampled]
    
    # Target chunk
    a_rest = topm_chunks[:, 0, 1:, :].detach()
    target_chunk = torch.cat([a_first.unsqueeze(1), a_rest], dim=1)
    target_padded = pad_vector(target_chunk, policy.config.max_action_dim)
    
    # Distill forward
    losses = policy.model.forward(images, img_masks, lang_tokens, lang_masks, state_p, target_padded)
    bc_loss = losses[:, 0, :7].mean()
    
    lora_l2 = 0.0
    for n, p in policy.named_parameters():
        if p.requires_grad and "lora_B" in n:
            lora_l2 = lora_l2 + p.pow(2).sum()
    total_loss = bc_loss + LORA_L2_COEF * lora_l2
    
    policy_optimizer.zero_grad()
    total_loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(
        [p for p in policy.parameters() if p.requires_grad], max_norm=1.0
    )
    policy_optimizer.step()
    
    with torch.no_grad():
        q_demo_mean = critic.q1(obs_critic, action_demo).mean().item()
        q_pol_mean  = critic.q1(obs_critic, a_first).mean().item()
    
    return {
        "bc_loss":   bc_loss.item(),
        "lora_l2":   (lora_l2.item() if torch.is_tensor(lora_l2) else lora_l2),
        "q_demo":    q_demo_mean,
        "q_pol":     q_pol_mean,
        "q_top":     q_top_mean,
        "q_best":    q_best_mean,
        "q_std":     q_std,
        "grad_norm": grad_norm.item(),
    }


## 9. Eval (same как offline)

In [9]:
@torch.no_grad()
def predict_action(raw_obs, task_text):
    obs = libero_obs_to_lerobot(raw_obs, task_text)
    obs = preprocessor(obs)
    a = policy.select_action(obs)
    a = postprocessor(a)
    return a.squeeze(0).detach().cpu().numpy().astype(np.float32)


def eval_libero(label=""):
    policy.eval()
    n_success = 0
    all_actions = []
    t_eval_start = time.time()
    for state_id in INIT_STATES_IDS:
        try:
            policy.reset()
            env.reset()
            raw_obs = env.set_init_state(init_states[state_id])
            success = False
            ep_actions = []
            for t in range(MAX_STEPS + NUM_STABILIZATION_STEPS):
                if t < NUM_STABILIZATION_STEPS:
                    action_list = [0.0]*6 + [-1.0]
                else:
                    a = predict_action(raw_obs, task_description_libero)
                    action_list = a.tolist()
                    ep_actions.append(a)
                raw_obs, _, done, _ = env.step(action_list)
                if done:
                    success = True
                    break
            if success: n_success += 1
            all_actions.extend(ep_actions)
            elapsed = time.time() - t_eval_start
            print(f"    state {state_id}: {'OK' if success else '..'} ({t+1} steps, {elapsed:.0f}s)", flush=True)
        except Exception as e:
            print(f"    state {state_id}: ERROR - {type(e).__name__}: {str(e)[:80]}", flush=True)
            continue
    
    all_actions = np.array(all_actions) if all_actions else np.zeros((1, 7))
    sr = n_success / len(INIT_STATES_IDS)
    mean_abs = np.abs(all_actions).mean(axis=0) if len(all_actions) else np.zeros(7)
    print(f"  [{label}] SR: {n_success}/13 = {sr:.0%}, action mean abs: {mean_abs.round(3)}", flush=True)
    policy.train()
    return n_success, sr, mean_abs


def save_checkpoint(path):
    os.makedirs(path, exist_ok=True)
    policy.save_pretrained(path)
    import shutil
    src_proc = MODEL_ID.replace("/", "--")
    hf_cache = os.path.expanduser(f"~/.cache/huggingface/hub/models--{src_proc}/snapshots")
    if os.path.exists(hf_cache):
        snapshot_dirs = [d for d in os.listdir(hf_cache) if not d.startswith(".")]
        if snapshot_dirs:
            proc_src = os.path.join(hf_cache, snapshot_dirs[0])
            for fname in os.listdir(proc_src):
                if fname.startswith("policy_"):
                    shutil.copy2(os.path.join(proc_src, fname), os.path.join(path, fname))


## 10. ONLINE training loop

Каждый шаг делает:
1. (Если step % ROLLOUT_EVERY == 0) — собрать N_ROLLOUTS_PER_COLLECTION rollouts
2. CRITIC_UPDATES_PER_DISTILL шагов IQL/Cal-QL critic update (если online buffer непустой)
3. 1 шаг PA-RL distillation (sample → global → local → distill BC)
4. (На EVAL_AT_STEPS) — eval + save best checkpoint


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [10]:
# Param groups
lora_params  = [p for n, p in policy.named_parameters() if "lora_" in n and p.requires_grad]
other_params = [p for n, p in policy.named_parameters() if "lora_" not in n and p.requires_grad]
policy_optimizer = optim.AdamW(
    [{"params": lora_params,  "weight_decay": 0.05},
     {"params": other_params, "weight_decay": 1e-6}],
    lr=POLICY_LR,
)
print(f"Policy optimizer: AdamW lr={POLICY_LR}, LoRA={sum(p.numel() for p in lora_params)/1e6:.2f}M")


def sample_batch_for_distill(batch_size, online_buffer, mixing_ratio):
    """Mixed sample: mixing_ratio из online, остальное offline."""
    n_online = int(batch_size * mixing_ratio)
    n_offline = batch_size - n_online
    if online_buffer.size < n_online:
        n_online = 0
        n_offline = batch_size
    
    if n_online == 0:
        return sample_offline_batch_for_distill(batch_size)
    
    online_batch  = online_buffer.sample(n_online)
    offline_batch = sample_offline_batch_for_distill(n_offline)
    
    # Объединяем (только distill fields — critic update идёт отдельно)
    merged = {}
    for k in ["img1_critic", "img2_critic", "state_critic",
              "img1_policy", "img2_policy", "state_raw", "action_demo"]:
        merged[k] = torch.cat([online_batch[k], offline_batch[k]], dim=0)
    return merged


print("="*70)
print(f"ONLINE PA-RL: эвалы на {EVAL_AT_STEPS}")
print(f"  Rollouts: каждые {ROLLOUT_EVERY} шагов, {N_ROLLOUTS_PER_COLLECTION} эпизодов")
print(f"  Critic updates: {CRITIC_UPDATES_PER_DISTILL} per distill step")
print(f"  Mixing: {MIXING_RATIO_DISTILL*100:.0f}% online / {(1-MIXING_RATIO_DISTILL)*100:.0f}% offline")
print("="*70)

eval_results = {}
best_n_success = -1
best_step = -1
SAVE_BEST_PATH = SAVE_VLA_PATH + "_best"

# ── Initial collection (чтобы critic было на чём учиться с шага 1) ──
print(f"\n[init] Собираем initial online rollouts ({N_ROLLOUTS_PER_COLLECTION} эп)…")
t0 = time.time()
n_succ_init, n_frames_init = collect_online_rollouts(n_rollouts=N_ROLLOUTS_PER_COLLECTION, buffer=online_buffer)
print(f"  ✓ {n_succ_init}/{N_ROLLOUTS_PER_COLLECTION} success, {n_frames_init} frames, {(time.time()-t0)/60:.1f}m. "
      f"Online buffer size: {online_buffer.size}")

# # ── Eval baseline (step 0) ──
# print(f"\n--- step 0 (baseline) ---")
# n_succ, sr, mean_abs = eval_libero(label="step 0")
# eval_results[0] = (n_succ, sr, mean_abs)
# if n_succ > best_n_success:
#     best_n_success = n_succ
#     best_step = 0
#     save_checkpoint(SAVE_BEST_PATH)
#     print(f"  ✓ BEST so far: {n_succ}/13 (saved)")

# ── Main loop ──
prev_step = 0
t0_total = time.time()
recent_metrics = {"bc": [], "td": [], "q_gap_distill": [], "q_demo_critic": []}

for next_eval_step in EVAL_AT_STEPS:
    if next_eval_step == 0:
        continue
    n_steps_to_train = next_eval_step - prev_step
    print(f"\n--- training {prev_step}->{next_eval_step} ({n_steps_to_train} steps) ---")
    
    for step in range(n_steps_to_train):
        absolute_step = prev_step + step + 1
        
        # === 1. Rollout collection ===
        if absolute_step % ROLLOUT_EVERY == 0:
            t_roll = time.time()
            n_succ_r, n_frames_r = collect_online_rollouts(
                n_rollouts=N_ROLLOUTS_PER_COLLECTION, buffer=online_buffer
            )
            print(f"  [step {absolute_step:3d}] collected {n_succ_r}/{N_ROLLOUTS_PER_COLLECTION} succ, "
                  f"{n_frames_r} frames, {(time.time()-t_roll)/60:.1f}m. "
                  f"buffer={online_buffer.size}", flush=True)
        
        # === 2. Critic updates: MIXED batches (Cal-QL practice, paper § 4.3) ===
        critic_metrics = None
        n_online_critic = int(BATCH_SIZE * MIXING_RATIO_DISTILL)
        n_offline_critic = BATCH_SIZE - n_online_critic
        if online_buffer.size < max(n_online_critic, 8):
            n_online_critic = 0
            n_offline_critic = BATCH_SIZE
        
        for _ in range(CRITIC_UPDATES_PER_DISTILL):
            online_part = online_buffer.sample(n_online_critic) if n_online_critic > 0 else None
            offline_part = sample_offline_batch_for_critic(n_offline_critic) if n_offline_critic > 0 else None
            
            if online_part is None:
                merged = offline_part
            elif offline_part is None:
                merged = online_part
            else:
                merged = {}
                for k in ["img1_curr", "img2_curr", "state_curr", "action",
                          "reward", "done", "img1_next", "img2_next", "state_next"]:
                    merged[k] = torch.cat([online_part[k], offline_part[k]], dim=0)
            
            critic_metrics = online_critic_step(merged)
        
        # === 3. PA-RL policy distill (mixed batch) ===
        distill_batch = sample_batch_for_distill(BATCH_SIZE, online_buffer, MIXING_RATIO_DISTILL)
        m = pa_rl_step(distill_batch)
        
        recent_metrics["bc"].append(m["bc_loss"])
        recent_metrics["q_gap_distill"].append(m["q_pol"] - m["q_demo"])
        if critic_metrics is not None:
            recent_metrics["td"].append(critic_metrics["td_loss"])
            recent_metrics["q_demo_critic"].append(critic_metrics["q_demo"])
        for k in recent_metrics:
            if len(recent_metrics[k]) > 10:
                recent_metrics[k].pop(0)
        
        # Log
        if absolute_step % LOG_EVERY == 0 or step == 0:
            elapsed = time.time() - t0_total
            cm_str = ""
            if critic_metrics is not None:
                cm_str = f" | td={critic_metrics['td_loss']:.2f} q_d_c={critic_metrics['q_demo']:.1f}"
            print(f"  step {absolute_step:3d} | bc={m['bc_loss']:.3f} | "
                  f"q_demo={m['q_demo']:.1f} q_pol={m['q_pol']:.1f} (gap={m['q_pol']-m['q_demo']:+.2f})"
                  f"{cm_str} | grad={m['grad_norm']:.2f} | buf={online_buffer.size} | "
                  f"t={elapsed/60:.1f}m", flush=True)
    
    # Eval
    avg_bc   = sum(recent_metrics["bc"]) / max(len(recent_metrics["bc"]), 1)
    avg_qgap = sum(recent_metrics["q_gap_distill"]) / max(len(recent_metrics["q_gap_distill"]), 1)
    avg_td   = sum(recent_metrics["td"]) / max(len(recent_metrics["td"]), 1) if recent_metrics["td"] else 0
    elapsed = time.time() - t0_total
    print(f"  trained. avg bc={avg_bc:.3f} q_gap_distill={avg_qgap:+.2f} td={avg_td:.2f} "
          f"buf={online_buffer.size}, elapsed={elapsed/60:.1f}min")
    
    n_succ, sr, mean_abs = eval_libero(label=f"step {next_eval_step}")
    eval_results[next_eval_step] = (n_succ, sr, mean_abs)
    
    if n_succ > best_n_success:
        best_n_success = n_succ
        best_step = next_eval_step
        print(f"  ✓ NEW BEST: {n_succ}/13 at step {best_step} (saving)")
        save_checkpoint(SAVE_BEST_PATH)
    elif n_succ < best_n_success - 2:
        print(f"  ⚠ Падение SR на 2+ от best. Останавливаемся.")
    
    prev_step = next_eval_step


# === Итоги ===
print("\n" + "="*70)
print("ИТОГИ — ONLINE PA-RL")
print("="*70)
print(f"\n{'step':<8} {'SR':<14} {'x':<8} {'y':<8} {'z':<8} {'gripper':<10}")
for step in sorted(eval_results.keys()):
    n_succ, sr, mean_abs = eval_results[step]
    is_best = " <-- BEST" if step == best_step else ""
    print(f"{step:<8} {n_succ}/13 ({sr:.0%})   "
          f"{mean_abs[0]:<8.3f} {mean_abs[1]:<8.3f} {mean_abs[2]:<8.3f} {mean_abs[6]:<10.3f}{is_best}")

print(f"\nBest checkpoint: step {best_step}, SR = {best_n_success}/13")
print(f"Сохранён в: {SAVE_BEST_PATH}")
print(f"Final online buffer size: {online_buffer.size}")

baseline_succ = eval_results[0][0]
if best_n_success > baseline_succ:
    print(f"\n✓ Online PA-RL улучшил SR на +{best_n_success - baseline_succ} эпизод(а) vs baseline ({baseline_succ}/13).")
elif best_n_success == baseline_succ:
    print(f"\n○ Online PA-RL = baseline.")
else:
    print(f"\n⚠ Online PA-RL хуже baseline.")


Policy optimizer: AdamW lr=1e-05, LoRA=4.42M
ONLINE PA-RL: эвалы на [20, 40, 60, 80, 100, 120, 140, 160]
  Rollouts: каждые 10 шагов, 5 эпизодов
  Critic updates: 1 per distill step
  Mixing: 50% online / 50% offline

[init] Собираем initial online rollouts (5 эп)…


  ✓ 4/5 success, 378 frames, 2.7m. Online buffer size: 378

--- training 0->20 (20 steps) ---


  step   1 | bc=0.744 | q_demo=-36.8 q_pol=-36.3 (gap=+0.43) | td=27.90 q_d_c=-35.9 | grad=0.14 | buf=378 | t=0.1m


  step   5 | bc=0.766 | q_demo=-35.4 q_pol=-34.8 (gap=+0.55) | td=3.53 q_d_c=-38.7 | grad=0.13 | buf=378 | t=0.4m


  [step  10] collected 4/5 succ, 385 frames, 3.7m. buffer=763


...

    state 47: OK (97 steps, 442s)


    state 46: .. (100 steps, 482s)


  [step 160] SR: 9/13 = 69%, action mean abs: [0.41  0.203 0.471 0.024 0.04  0.036 1.001]


  ⚠ Падение SR на 2+ от best. Останавливаемся.

ИТОГИ — ONLINE PA-RL

step     SR             x        y        z        gripper   
20       5/13 (38%)   0.468    0.180    0.530    0.993     
40       5/13 (38%)   0.447    0.188    0.522    0.994     
60       8/13 (62%)   0.463    0.186    0.522    0.997     
80       7/13 (54%)   0.454    0.188    0.508    0.993     
100      12/13 (92%)   0.451    0.201    0.520    1.001      <-- BEST
120      10/13 (77%)   0.429    0.198    0.498    0.999     
140      4/13 (31%)   0.427    0.202    0.481    0.998     
160      9/13 (69%)   0.410    0.203    0.471    1.001     

Best checkpoint: step 100, SR = 12/13
Сохранён в: /workspace/out/smolvla_parl_online_best
Final online buffer size: 5000


KeyError: 0